# Assignment 4: Phrase Queries and PageRank

**Student names**: Mustafa <br>
**Group number**: 125 <br>
**Date**: 30.10.2026 23:59

In this assignment, we will build indexes for phrase queries, compute PageRank, and answer conceptual questions.

## Dataset and preprocessing

Place `cran.all.1400` next to this notebook. The parser below loads its 1,400 documents into `docs`.
Use only each document's `abstract` field.

In [ ]:
# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"cran.all.1400" 

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)

### Preprocessing

Implement `tokenize(text)` and tokenize the abstracts. You may reuse your tokenizer from an earlier assignment, but this time keep the stopwords.

In [ ]:
# TODO: Implement preprocessing
import re

def tokenize(text: str) -> list[str]:
    # Your code here
    raise NotImplementedError("Implement tokenization")

token_lists = {doc_id: tokenize(doc["abstract"]) for doc_id, doc in docs.items()}

## 4.1 Indexes for phrase queries

### 4.1.1 Biword index

A biword is a pair of consecutive tokens. Implement:

- `build_biword_index(token_lists)`: return a dictionary mapping each biword, written as two tokens separated by one space, to sorted, unique document IDs.
- `biword_phrase_search(phrase, index)`: tokenize the phrase and return the sorted IDs of documents containing all its biwords. You may assume at least two query tokens.

In [ ]:
# TODO: Build and search the biword index
def build_biword_index(token_lists):
    # Your code here
    raise NotImplementedError("Implement the biword index")

def biword_phrase_search(phrase, index):
    # Your code here
    raise NotImplementedError("Implement biword search")

biword_index = build_biword_index(token_lists)

### 4.1.2 Positional index

Implement `build_positional_index(token_lists)` with the structure:

```python
index[term][doc_id] = [position_1, position_2, ...]
```

Store every occurrence, with positions starting at 1 within each document. Keep document IDs and position lists in ascending order.

Then implement `phrase_search(phrase, index)` using the positional index. Return sorted, unique IDs of documents containing the complete phrase.

Return `[]` for an empty query or a query token absent from the index.

In [ ]:
# TODO: Build the positional index and implement phrase search
def build_positional_index(token_lists):
    # Your code here
    raise NotImplementedError("Implement the positional index")

def phrase_search(phrase, index):
    # Your code here
    raise NotImplementedError("Implement phrase search")

positional_index = build_positional_index(token_lists)

### 4.1.3 Run phrase queries

Run the following queries with both indexes. The cell prints the number of returned documents and the first five IDs.

In [ ]:
phrase_queries = [
    "boundary layer",
    "laminar boundary layer",
    "free stream mach number",
    "in the wind tunnel",
]

for query in phrase_queries:
    biword_docs = biword_phrase_search(query, biword_index)
    phrase_docs = phrase_search(query, positional_index)
    print(f"{query!r}")
    print(f"  Biword:     {len(biword_docs):3d} documents; first five: {biword_docs[:5]}")
    print(f"  Positional: {len(phrase_docs):3d} documents; first five: {phrase_docs[:5]}")

## 4.2 PageRank

The graph in `painters.json` represents 14 Wikipedia pages about painters and 50 directed links between them. An arrow from A to B means that page A links to page B. We will rank the pages within this collection.

![Directed links between the painters. Numbers identify names in the legend.](painters.svg)

The provided cell loads the collection and creates the adjacency matrix `A` as a list of rows. Each row identifies the page a link comes from, and each column identifies the destination page. Thus, `A[i][j] = 1` means that page `i` links to page `j`. If there is no link, the entry is 0.

1. Implement `transition_matrix(A, alpha=0.15)`. Return a new transition matrix `P` with floating-point probabilities, as a list of rows or a NumPy array, where `P[i][j]` is the probability of moving from page `i` to page `j`. Each row must sum to 1.
   - If a page has outgoing links, distribute probability `1 - alpha` equally among them, and probability `alpha` equally among **all** pages, including the current page.
   - If a page has no outgoing links (a dead end), distribute its entire probability equally among all pages.

   For example, Picasso links to Manet and Matisse. With `alpha=0.15`, split the 85% link-following probability between those two destinations, and distribute the remaining 15% across all fourteen pages.

2. Implement `pagerank(P, x0, iterations=50)`. Starting from the provided initial scores `x0`, repeatedly update the score vector using $x_{t+1}=x_tP$ (`x @ P` if you use NumPy arrays). Return the scores after `iterations` updates. This repeated update is called **power iteration**.

In the final cell, the transition matrix you constructed and the `pagerank`-function are used to rank the Wikipedia pages.

In [ ]:
import json

with open("painters.json", encoding="utf-8") as f:
    painters = json.load(f)

pages = painters["names"]
n = len(pages)
A = [[0] * n for _ in range(n)]
for source, target in painters["links"]:
    A[source][target] = 1

x0 = [1 / n] * n


In [ ]:
# TODO: Construct the transition matrix and compute PageRank
def transition_matrix(A, alpha=0.15):
    # Your code here
    raise NotImplementedError("Implement the transition matrix")

def pagerank(P, x0, iterations=50):
    # Your code here
    raise NotImplementedError("Implement PageRank")

In [ ]:
P = transition_matrix(A, alpha=0.15)
scores = pagerank(P, x0)
incoming_links = [sum(A[i][j] for i in range(n)) for j in range(n)]
for i in sorted(range(n), key=lambda i: scores[i], reverse=True):
    print(f"{pages[i]:24} PageRank: {scores[i]:.5f}   Incoming links: {incoming_links[i]}")


## 4.3 Conceptual questions

Answer briefly, using your results where relevant.

1. Why can the biword index return extra documents for longer phrases? How does the positional index avoid these matches?
2. Does the painter with the highest PageRank also have the most incoming links? Explain what PageRank captures beyond counting incoming links.
3. Why does PageRank include teleportation?
4. Why might a crawler fetch a lower-priority URL from one host before a higher-priority URL from another host?

*Your answers here*